In [ ]:

import os
import cv2
import gc
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from tqdm.notebook import tqdm
from shapely.geometry import Polygon
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import shutil

DATA_ROOT = ''
WORK_DIR = ''


PATCH_SIZE = 512
STRIDE = 512        
MIN_TISSUE = 0.05   # 빈 패치 드랍
BATCH_SIZE = 32     
EPOCHS = 30         

# Create Directories
for split in ['train', 'val']:
    os.makedirs(f"{WORK_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/{split}/labels", exist_ok=True)

In [3]:
import torch
print("torch:", torch.__version__)
print("torch cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.9.1+cu128
torch cuda available: True
torch cuda version: 12.8
gpu: NVIDIA GeForce RTX 4060 Ti


In [4]:
def rle2mask(mask_rle, shape):
    """Decodes RLE string to numpy array."""
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)                                      #결측데이터면 마스크 전부 0으로 반환
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]  #짝수번째와 홀수번째를 각각 starts, length로 나눔 (1을 쓸곳만 기억)
    starts -= 1                                                                     #RLE에서 시작이 1이지만 numpy에서는 0이어서 1을 빼줌
    ends = starts + lengths                                                         #starts에서 length만큼 1을 입력하는거니까 1이 끝나는 지점을 의미
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)                               #1차원 배열로 만들고
    for lo, hi in zip(starts, ends):                                                #RLE값만큼 1을 입력함
        img[lo:hi] = 1
    return img.reshape(shape, order='F')                                            #그 후 2차원 구조로 변환

def tissue_ratio(patch):
    """Calculates the percentage of tissue in a patch."""
    if len(patch.shape) == 2 or patch.shape[-1] == 1:
        gray = patch
    else:
        gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    
    # Otsu's thresholding
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)              #임계값 기준으로 흑,백 두가지로 나눔
    # Count non-background pixels (assuming background is light/white in H&E usually, 
    # but Otsu finds the separator. For HuBMAP, background is often white).
    # Inverted check: tissue is usually darker.
    tissue_pixels = np.sum(mask == 0)       
    return tissue_pixels / mask.size             #조직 내 흰색 비율을 반환(나중에 MIN_TISSUE랑 비교)

def mask_to_polygons(mask):
    """Converts binary mask to normalized polygons for YOLO."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) #마스크 윤곽선 추출
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 200: # 범위가 너무 작은것들은 드랍
            continue
        cnt = cnt.squeeze()
        if len(cnt.shape) < 2 or len(cnt) < 6: # 폴리곤의 점이 너무 적거나 선분일형태일때 드랍
            continue
        polygons.append(cnt)
    return polygons

In [5]:
# 데이터 불러오기
df_masks = pd.read_csv(f"{DATA_ROOT}/train.csv").set_index('id')
image_ids = [f.split(".")[0] for f in os.listdir(f"{DATA_ROOT}/train") if f.endswith(".tiff")]

# 이미지 ID별로 split
train_ids, val_ids = train_test_split(image_ids, test_size=0.1, random_state=42)
split_map = {img_id: 'train' for img_id in train_ids}
split_map.update({img_id: 'val' for img_id in val_ids})

print(f"Processing {len(train_ids)} Train images and {len(val_ids)} Validation images...")

for img_id in tqdm(image_ids):
    if img_id not in df_masks.index: continue
    
    split = split_map[img_id]
    img_path = f"{DATA_ROOT}/train/{img_id}.tiff"
    
    with rasterio.open(img_path) as src:
        h, w = src.height, src.width
        # 이미지를 한개씩만
        rle = df_masks.loc[img_id, 'encoding']
        full_mask = rle2mask(rle, (h, w))   #이미지 높이너비에 맞춰 마스크 생성
        
        # Stride 크기씩 스캔
        for y in range(0, h - PATCH_SIZE, STRIDE):
            for x in range(0, w - PATCH_SIZE, STRIDE):
                
                # 1. Read Patch
                window = Window(x, y, PATCH_SIZE, PATCH_SIZE)   #영역 지정
                patch = src.read(window=window)
                patch = np.moveaxis(patch, 0, -1) # (C,H,W) -> (H,W,C)
                
                # 2. Tissue Check (빈공간 스킵)
                if tissue_ratio(patch) < MIN_TISSUE:
                    continue
                
                # 3. Handle Mask Patch 현재 위치에 맞춰 마스크 잘라오기
                mask_patch = full_mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                
                # 4. Prepare Filename
                fname = f"{img_id}_{x}_{y}"
                save_path_img = f"{WORK_DIR}/{split}/images/{fname}.jpg"
                save_path_lbl = f"{WORK_DIR}/{split}/labels/{fname}.txt"
                
                # 5. Save Image (Force BGR for OpenCV/YOLO)
                if len(patch.shape) == 2 or patch.shape[-1] == 1:
                    save_img = cv2.cvtColor(patch, cv2.COLOR_GRAY2BGR)
                else:
                    save_img = cv2.cvtColor(patch, cv2.COLOR_RGB2BGR)
                
                cv2.imwrite(save_path_img, save_img)
                
                # 6. 폴리곤 값으로 라벨생성
                polys = mask_to_polygons(mask_patch)
                with open(save_path_lbl, "w") as f:
                    if polys:
                        for poly in polys:
                            # Normalize coordinates (0-1)
                            poly = poly.astype(float)
                            poly[:, 0] /= PATCH_SIZE
                            poly[:, 1] /= PATCH_SIZE
                            
                            # Flatten and write
                            coords = " ".join(map(str, poly.flatten()))
                            f.write(f"0 {coords}\n")
                    # If no polys, create empty file (background image)
                    
        # Cleanup memory immediately
        del full_mask
        gc.collect()

Processing 13 Train images and 2 Validation images...


  0%|          | 0/15 [00:00<?, ?it/s]

c:\Users\hyper\anaconda3\envs\yolopr\Lib\site-packages\rasterio\__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [6]:
yaml_content = f"""
path: {WORK_DIR}
train: train/images
val: val/images

nc: 1
names: ['glomerulus']
"""

with open(f"{WORK_DIR}/data.yaml", "w") as f:
    f.write(yaml_content)

print("Data YAML created.")

Data YAML created.


In [7]:
import torch; assert torch.cuda.is_available(), "CUDA(GPU) not available"; print("✅ GPU:", torch.cuda.get_device_name(0))


✅ GPU: NVIDIA GeForce RTX 4060 Ti


In [8]:
# 모델
model = YOLO("yolo11n-seg.pt") 

# 학습
results = model.train(
    data=f"{WORK_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=PATCH_SIZE,
    batch=BATCH_SIZE,      
    patience=5,            
    optimizer="AdamW",
    lr0=1e-3,
    augment=True,
    workers=4,
    device = 0,             
    project="hubmap_project",
    name="run_v1",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.243 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.11.14 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\hyper\Downloads\\hubmap-kidney-segmentation\work/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_

In [9]:
# Run validation on the separate validation set
metrics = model.val()
print(f"mAP50-95: {metrics.box.map}")
print(f"Seg mAP50-95: {metrics.seg.map}")

# Cleanup (Optional: deletes images to save space for commit)
# shutil.rmtree(WORK_DIR)

Ultralytics 8.3.241  Python-3.11.14 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
YOLO11n-seg summary (fused): 113 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 629.6491.8 MB/s, size: 64.4 KB)
val: Scanning C:\Users\hyper\Downloads\플젝\hubmap-kidney-segmentation\work\val\labels.cache... 10793 images, 9411 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10793/10793  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 675/675 17.9it/s 37.7s<0.1s
                   all      10793       1707      0.911      0.849      0.912      0.721      0.909      0.847      0.907      0.692
Speed: 0.8ms preprocess, 1.7ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Users\hyper\Downloads\\runs\segment\val5
mAP50-95: 0.7211702365294865
Seg mAP50-95: 0.6921359522528726


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from pathlib import Path

# === 1) 기본 설정 ===
DATA_ROOT = Path(DATA_ROOT)  # 기존 yolov11.ipynb 변수(DATA_ROOT) 사용 중이면 그대로
df = pd.read_csv(DATA_ROOT / "train.csv").set_index("id")  # encoding 읽기용

def to_uint8(img):
    img = np.asarray(img)
    if img.dtype == np.uint8:
        return img
    if img.dtype == np.uint16:
        return (img / 256).astype(np.uint8)
    mn, mx = np.nanmin(img), np.nanmax(img)
    if mx <= mn:
        return np.zeros_like(img, dtype=np.uint8)
    return ((img - mn) / (mx - mn) * 255).astype(np.uint8)

# === 2) rasterio로 "처음부터" 다운샘플 이미지 읽기 ===
def read_tiff_downsample_rasterio(tiff_path, scale=20):
    with rasterio.open(str(tiff_path)) as src:
        H, W = src.height, src.width
        out_h, out_w = H // scale, W // scale

        img = src.read(
            out_shape=(src.count, out_h, out_w),
            resampling=Resampling.bilinear
        )  # (C, h, w)

    img = np.moveaxis(img, 0, -1)  # (h,w,C)
    img = to_uint8(img)

    # 채널 정리(시각화용)
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    elif img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    elif img.shape[2] > 3:
        img = img[:, :, :3]

    return img, (H, W)

# === 3) RLE -> 다운샘플 마스크 (full mask 생성 없이) ===
def rle_to_downsample_mask(mask_rle, H, W, scale=20):
    out_h, out_w = H // scale, W // scale
    small = np.zeros((out_h, out_w), dtype=np.uint8)

    if mask_rle is None or (isinstance(mask_rle, float) and np.isnan(mask_rle)) or str(mask_rle).strip() == "":
        return small

    s = str(mask_rle).split()
    starts = np.asarray(s[0::2], dtype=np.int64) - 1  # 0-based
    lengths = np.asarray(s[1::2], dtype=np.int64)
    ends = starts + lengths  # exclusive

    # RLE는 column-major: index -> (col, row) with divmod(idx, H)
    for st, en in zip(starts, ends):
        c_start, r_start = divmod(st, H)
        c_end,   r_end   = divmod(en - 1, H)

        # 이 run이 덮는 컬럼들을 순회 (보통 mask 영역만이라 생각보다 빠름)
        for c in range(c_start, c_end + 1):
            x = c // scale
            if x < 0 or x >= out_w:
                continue

            rr0 = r_start if c == c_start else 0
            rr1 = r_end   if c == c_end   else (H - 1)

            y0 = rr0 // scale
            y1 = rr1 // scale

            if y1 < 0 or y0 >= out_h:
                continue

            # clip (floor 다운샘플이므로 out_h/out_w 범위에 맞춤)
            y0 = max(y0, 0)
            y1 = min(y1, out_h - 1)

            small[y0:y1+1, x] = 1

    return small

# === 4) 다운샘플 오버레이 시각화 ===
def show_downsample_overlay(image_id, scale=20, alpha=0.5):
    tiff_path = DATA_ROOT / "train" / f"{image_id}.tiff"
    img_small, (H, W) = read_tiff_downsample_rasterio(tiff_path, scale=scale)

    rle = df.loc[image_id, "encoding"] if image_id in df.index else None
    mask_small = rle_to_downsample_mask(rle, H, W, scale=scale)

    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_small)
    plt.title(f"{image_id} downsample x{scale}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_small)
    plt.imshow(mask_small, cmap="hot", alpha=alpha)
    plt.title(f"Overlay (mask) downsample x{scale}")
    plt.axis("off")

    plt.show()

# === 5) 사용 예시 ===
small_ids = ["0486052bb"]  # 너가 쓰던 리스트
for sid in small_ids:
    show_downsample_overlay(sid, scale=20, alpha=0.5)


c:\Users\hyper\anaconda3\envs\yolopr\Lib\site-packages\rasterio\__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


<Figure size 1400x600 with 2 Axes>

In [11]:
small_ids = ["0486052bb"]  # 너가 쓰던 리스트
for sid in small_ids:
    show_downsample_overlay(sid, scale=20, alpha=0.5)

<Figure size 1400x600 with 2 Axes>